In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM
from trl import SFTTrainer, SFTConfig

In [30]:
model_name = "data"
dataset_name = "data"
device_map = 'auto'

In [27]:
peft_config = LoraConfig(
      lora_alpha="",
      lora_dropout="",
      r="",
      bias="",
      task_type="",
)

In [28]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit="",
    bnb_4bit_use_double_quant="",
    bnb_4bit_quant_type="",
    bnb_4bit_compute_dtype=""
)

In [29]:
def prompt_instruction_format(sample):
  return f"""### Instruction:

    ### Task:
    {sample['']}

    ### Input:
    {sample['']}

    ### Response:
    {sample['']}
    """

In [18]:
dataset = load_dataset(dataset_name, split='train')

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name,
          quantization_config="",
          use_cache = "",
          device_map="")
model.config.pretraining_tp = 1

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = ""

In [39]:
training_args = SFTConfig(
    output_dir="",
    learning_rate="",
    weight_decay="",
    lr_scheduler_type="",
    disable_tqdm="",
    seed=42,
    max_seq_length="",
    packing="",
    report_to="",
    neftune_noise_alpha=""
    )

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    formatting_func=prompt_instruction_format,
    args=training_args,
)

In [ ]:
trainer.train()

In [42]:
trainer.save_model('')

In [ ]:
trained_model = AutoModelForCausalLM.from_pretrained('')
trained_tokenizer = AutoTokenizer.from_pretrained('')

In [50]:
instruction =""
input =""

prompt = f"""### Instruction:
Use the Task below and the Input given to write the Response, which is a programming code that can solve the Task.

### Task:
{instruction}

### Input:
{input}

### Response:
"""

input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids
outputs = trained_model.generate(input_ids=input_ids, max_new_tokens=200, do_sample=True, top_p=0.9,temperature=0.7)

In [ ]:
print(f"{prompt}\n")
print(f"{trained_tokenizer.batch_decode(outputs.cpu().numpy(), skip_special_tokens=True)[0][len(prompt):].sptrip()}")